In [1]:
# ============================================================
# D10 — Branch C: Normalised Markdown conversion
# 0. Imports and frozen experimental configuration
# ============================================================

import json
import hashlib
import platform
import re
import sys
import unicodedata

from collections import Counter
from datetime import datetime
from pathlib import Path

from google.colab import files

DOCUMENT_ID = "D10"
DOCUMENT_NAME = "IMPI — Inquérito Mensal à Produção Industrial"

BRANCH = "C"
BRANCH_NAME = "Deterministic normalisation"
PARENT_BRANCH = "B"

SOURCE_FORMAT = ".pdf"
EXPECTED_PAGE_COUNT = 4

EXPECTED_SOURCE_SHA256 = (
    "fb72aac548f61578bc9ba52448793b81bf61e37eff146f9519124d0794a4f4e5"
)

EXPECTED_BRANCH_B_REPRESENTATION_SHA256 = (
    "8a550395d9e24ceab8bca51e885658c7aca4a9dbae1480758f8aefe82f3cb81b"
)

# ------------------------------------------------------------
# Frozen Stage 1 expectations.
# Used only AFTER extraction for diagnostics / Stage 4 validation.
# They are NOT disclosed to the model.
# ------------------------------------------------------------

EXPECTED_RECORD_COUNT = 69

EXPECTED_CATEGORY_COUNTS = {
    "Instrument metadata": 8,
    "Questionnaire field": 32,
    "UAE template element": 6,
    "Product table field": 12,
    "Instruction": 11
}

EXPECTED_FIELDS = [
    "Category",
    "Section",
    "Field or Concept",
    "Description",
    "Code",
    "Expected Value Type",
    "Source Location"
]

ALLOWED_CATEGORIES = set(EXPECTED_CATEGORY_COUNTS)

EXPECTED_QUESTIONNAIRE_CODES = {
    "BC001",
    "BC005",
    "BC007",
    "BC010",
    "BC015",
    "BC020",
    "BC025",
    "BC030"
}

EXPECTED_UAE_TEMPLATE_LABELS = {
    "Código da UAE",
    "Designação da UAE",
    "Situação da UAE perante a atividade",
    "Observações da UAE",
    "Confirmar",
    "Produtos"
}

EXPECTED_PRODUCT_TABLE_FIELDS = {
    "NIF",
    "UAE",
    "Período de Referência",
    "Nº",
    "Produto",
    "Unid.",
    "Código",
    "Quantidades produzidas",
    "Quantidades vendidas",
    "Valor das vendas / prestação de serviços",
    "Observações empresa",
    "Observações INE"
}

CRITICAL_PARENT_MARKERS = {
    "survey_title":
        "IMPI - Inquérito Mensal à Produção Industrial",

    "registration_number":
        "10067",

    "validity_date":
        "2026/12/31",

    "section_i":
        "Identificação da unidade estatística",

    "section_ii":
        "Situação da unidade estatística",

    "section_iii":
        "Observações",

    "section_iv":
        "Responsável pelo preenchimento",

    "uae":
        "Unidade de Atividade Económica",

    "quantities_produced":
        "QUANTIDADES PRODUZIDAS",

    "quantities_sold":
        "QUANTIDADES VENDIDAS",

    "sales_value":
        "VALOR DAS VENDAS",

    "filling_instructions":
        "INSTRUÇÕES DE PREENCHIMENTO",

    "prodcom":
        "PRODCOM"
}

OUTPUT_DIR = Path("outputs_D10_branch_C")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PARENT_CHECK_PATH = (
    OUTPUT_DIR / "D10_branch_C_parent_B_equivalence_check.json"
)
NORMALISATION_CHECK_PATH = (
    OUTPUT_DIR / "D10_branch_C_normalisation_check.json"
)
REPRESENTATION_PATH = (
    OUTPUT_DIR / "D10_branch_C_normalised_markdown.md"
)
REPRESENTATION_METADATA_PATH = (
    OUTPUT_DIR / "D10_branch_C_representation_metadata.json"
)
PROMPT_PATH = (
    OUTPUT_DIR / "D10_branch_C_prompt.txt"
)
EXPERIMENT_METADATA_PRE_PATH = (
    OUTPUT_DIR / "D10_branch_C_experiment_metadata_pre.json"
)
PRECHECK_PATH = (
    OUTPUT_DIR / "D10_branch_C_pre_extraction_check.json"
)
RAW_RESPONSE_PATH = (
    OUTPUT_DIR / "D10_branch_C_raw_response.txt"
)
PARSED_EXTRACTION_PATH = (
    OUTPUT_DIR / "D10_branch_C_parsed_extraction.json"
)
STRUCTURE_CHECK_PATH = (
    OUTPUT_DIR / "D10_branch_C_structure_check.json"
)
EXPERIMENT_METADATA_PATH = (
    OUTPUT_DIR / "D10_branch_C_experiment_metadata.json"
)
EXPERIMENT_SUMMARY_PATH = (
    OUTPUT_DIR / "D10_branch_C_experiment_summary.json"
)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Parent branch:", PARENT_BRANCH)
print("Expected physical pages:", EXPECTED_PAGE_COUNT)


Document: D10
Branch: C
Parent branch: B
Expected physical pages: 4


In [2]:
# ============================================================
# 1. Upload original D10 PDF and required frozen Branch B artefacts
# ============================================================
#
# Upload exactly:
#   1) original D10 PDF
#   2) D10_branch_B_structural_markdown.md
#   3) D10_branch_B_conversion_integrity.json
#
# Branch C starts from the frozen Branch B representation.
# ============================================================

uploaded = files.upload()
names = list(uploaded.keys())

pdf_files = [
    Path(name)
    for name in names
    if name.lower().endswith(".pdf")
]

md_files = [
    Path(name)
    for name in names
    if name.lower().endswith(".md")
]

json_files = [
    Path(name)
    for name in names
    if name.lower().endswith(".json")
]

if (
    len(pdf_files) != 1
    or len(md_files) != 1
    or len(json_files) != 1
):
    raise ValueError(
        "Upload exactly one D10 PDF, one frozen Branch B structural "
        "Markdown file, and one Branch B conversion-integrity JSON file."
    )

SOURCE_PATH = pdf_files[0]
BRANCH_B_REPRESENTATION_PATH = md_files[0]
BRANCH_B_CHECK_PATH = json_files[0]

print("Source:", SOURCE_PATH.name)
print("Branch B representation:", BRANCH_B_REPRESENTATION_PATH.name)
print("Branch B integrity:", BRANCH_B_CHECK_PATH.name)


Saving D10_branch_B_structural_markdown.md to D10_branch_B_structural_markdown.md
Saving D10_branch_B_conversion_integrity.json to D10_branch_B_conversion_integrity.json
Saving D10 - IMPI_QUESTIONARIO INE Portugal 2026.pdf to D10 - IMPI_QUESTIONARIO INE Portugal 2026.pdf
Source: D10 - IMPI_QUESTIONARIO INE Portugal 2026.pdf
Branch B representation: D10_branch_B_structural_markdown.md
Branch B integrity: D10_branch_B_conversion_integrity.json


In [3]:
# ============================================================
# 2. Verify frozen source identity and Branch B provenance
# ============================================================

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(chunk_size),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def sha256_text(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


if SOURCE_PATH.suffix.lower() != SOURCE_FORMAT:
    raise ValueError("Unexpected D10 source format.")


SOURCE_SHA256 = sha256_file(
    SOURCE_PATH
)

SOURCE_HASH_MATCH = (
    SOURCE_SHA256
    == EXPECTED_SOURCE_SHA256
)

if not SOURCE_HASH_MATCH:
    raise ValueError(
        "Uploaded D10 PDF does not match the frozen Stage 1 source identity."
    )


with open(
    BRANCH_B_CHECK_PATH,
    "r",
    encoding="utf-8"
) as f:
    branch_b_check = json.load(f)


if branch_b_check.get("document_id") != DOCUMENT_ID:
    raise ValueError(
        "Branch B conversion-integrity artefact belongs to another document."
    )

if branch_b_check.get("branch") != "B":
    raise ValueError(
        "Uploaded conversion-integrity artefact is not from Branch B."
    )

if branch_b_check.get("source_sha256") != SOURCE_SHA256:
    raise ValueError(
        "Branch B conversion-integrity artefact refers to another D10 source."
    )

if not branch_b_check.get(
    "conversion_integrity_passed",
    False
):
    raise ValueError(
        "The frozen D10 Branch B representation did not pass conversion integrity."
    )


SOURCE_B_MARKDOWN = (
    BRANCH_B_REPRESENTATION_PATH.read_text(
        encoding="utf-8"
    )
)

if not SOURCE_B_MARKDOWN.strip():
    raise ValueError(
        "Uploaded Branch B structural Markdown is empty."
    )


UPLOADED_BRANCH_B_SHA256 = sha256_text(
    SOURCE_B_MARKDOWN
)

BRANCH_B_HASH_MATCH = (
    UPLOADED_BRANCH_B_SHA256
    == EXPECTED_BRANCH_B_REPRESENTATION_SHA256
)

if not BRANCH_B_HASH_MATCH:
    raise ValueError(
        "Uploaded Branch B Markdown does not match the frozen final "
        "D10 Branch B representation SHA-256."
    )


print("Frozen source identity verified.")
print("Branch B conversion provenance verified.")
print("Frozen Branch B SHA-256 verified:", BRANCH_B_HASH_MATCH)


Frozen source identity verified.
Branch B conversion provenance verified.
Frozen Branch B SHA-256 verified: True


In [4]:
# ============================================================
# 3. Verify frozen Branch B parent equivalence
# ============================================================
#
# Branch C does not create another structural conversion.
# It receives the exact frozen Branch B representation and verifies:
# - frozen source identity;
# - passing B conversion integrity;
# - exact B representation hash;
# - four physical-page boundaries;
# - expected source content and questionnaire codes;
# - repeated UAE/sample-row content remains represented.
# ============================================================

PAGE_PATTERN = re.compile(
    r"^## Source Page (\d+)$",
    flags=re.MULTILINE
)

branch_b_page_markers = PAGE_PATTERN.findall(
    SOURCE_B_MARKDOWN
)

expected_page_markers = [
    str(page_number)
    for page_number in range(
        1,
        EXPECTED_PAGE_COUNT + 1
    )
]

BRANCH_B_PAGE_SEQUENCE_VALID = (
    branch_b_page_markers
    == expected_page_markers
)


canonical_parent = unicodedata.normalize(
    "NFKC",
    SOURCE_B_MARKDOWN
).casefold()

parent_marker_checks = {
    key:
        unicodedata.normalize(
            "NFKC",
            marker
        ).casefold()
        in canonical_parent
    for key, marker
    in CRITICAL_PARENT_MARKERS.items()
}

BRANCH_B_CRITICAL_MARKERS_VALID = all(
    parent_marker_checks.values()
)


observed_parent_codes = sorted(
    set(
        re.findall(
            r"\bBC\d{3}\b",
            SOURCE_B_MARKDOWN
        )
    )
)

BRANCH_B_QUESTIONNAIRE_CODES_VALID = (
    set(observed_parent_codes)
    == EXPECTED_QUESTIONNAIRE_CODES
)


sample_product_checks = {
    label:
        label in SOURCE_B_MARKDOWN
    for label in [
        "Produto a",
        "Produto b",
        "Produto c",
        "Produto d"
    ]
}

BRANCH_B_SAMPLE_ROWS_RETAINED = all(
    sample_product_checks.values()
)


# These are source instances, not expected extraction records.
repeated_uae_status_count_parent = len(
    re.findall(
        r"Situação da UAE perante a atividade",
        SOURCE_B_MARKDOWN,
        flags=re.IGNORECASE
    )
)

BRANCH_B_REPEATED_UAE_CONTENT_RETAINED = (
    repeated_uae_status_count_parent >= 3
)


PARENT_EQUIVALENCE_PASSED = bool(
    SOURCE_HASH_MATCH
    and branch_b_check.get(
        "conversion_integrity_passed",
        False
    )
    and BRANCH_B_HASH_MATCH
    and BRANCH_B_PAGE_SEQUENCE_VALID
    and BRANCH_B_CRITICAL_MARKERS_VALID
    and BRANCH_B_QUESTIONNAIRE_CODES_VALID
    and BRANCH_B_SAMPLE_ROWS_RETAINED
    and BRANCH_B_REPEATED_UAE_CONTENT_RETAINED
)


parent_check = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "parent_branch":
        PARENT_BRANCH,

    "source_sha256":
        SOURCE_SHA256,

    "source_hash_matches_frozen_identity":
        SOURCE_HASH_MATCH,

    "branch_B_conversion_integrity_passed":
        bool(
            branch_b_check.get(
                "conversion_integrity_passed",
                False
            )
        ),

    "expected_frozen_branch_B_sha256":
        EXPECTED_BRANCH_B_REPRESENTATION_SHA256,

    "uploaded_branch_B_sha256":
        UPLOADED_BRANCH_B_SHA256,

    "uploaded_branch_B_hash_matches_frozen_parent":
        BRANCH_B_HASH_MATCH,

    "branch_B_page_markers":
        branch_b_page_markers,

    "branch_B_page_sequence_valid":
        BRANCH_B_PAGE_SEQUENCE_VALID,

    "branch_B_critical_marker_checks":
        parent_marker_checks,

    "branch_B_critical_markers_valid":
        BRANCH_B_CRITICAL_MARKERS_VALID,

    "branch_B_questionnaire_codes":
        observed_parent_codes,

    "branch_B_questionnaire_codes_valid":
        BRANCH_B_QUESTIONNAIRE_CODES_VALID,

    "branch_B_sample_product_checks":
        sample_product_checks,

    "branch_B_sample_product_rows_retained":
        BRANCH_B_SAMPLE_ROWS_RETAINED,

    "branch_B_repeated_uae_status_count":
        repeated_uae_status_count_parent,

    "branch_B_repeated_uae_content_retained":
        BRANCH_B_REPEATED_UAE_CONTENT_RETAINED,

    "parent_equivalence_method":
        (
            "Frozen Branch B representation SHA-256 + Branch B "
            "conversion-integrity provenance; Branch B is not regenerated"
        ),

    "branch_B_regeneration_attempted":
        False,

    "parent_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED
}


PARENT_CHECK_PATH.write_text(
    json.dumps(
        parent_check,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        parent_check,
        ensure_ascii=False,
        indent=2
    )
)


if not PARENT_EQUIVALENCE_PASSED:
    raise ValueError(
        "D10 Branch C parent-equivalence verification failed."
    )


{
  "document_id": "D10",
  "branch": "C",
  "parent_branch": "B",
  "source_sha256": "fb72aac548f61578bc9ba52448793b81bf61e37eff146f9519124d0794a4f4e5",
  "source_hash_matches_frozen_identity": true,
  "branch_B_conversion_integrity_passed": true,
  "expected_frozen_branch_B_sha256": "8a550395d9e24ceab8bca51e885658c7aca4a9dbae1480758f8aefe82f3cb81b",
  "uploaded_branch_B_sha256": "8a550395d9e24ceab8bca51e885658c7aca4a9dbae1480758f8aefe82f3cb81b",
  "uploaded_branch_B_hash_matches_frozen_parent": true,
  "branch_B_page_markers": [
    "1",
    "2",
    "3",
    "4"
  ],
  "branch_B_page_sequence_valid": true,
  "branch_B_critical_marker_checks": {
    "survey_title": true,
    "registration_number": true,
    "validity_date": true,
    "section_i": true,
    "section_ii": true,
    "section_iii": true,
    "section_iv": true,
    "uae": true,
    "quantities_produced": true,
    "quantities_sold": true,
    "sales_value": true,
    "filling_instructions": true,
    "prodcom": true
  },

In [5]:
# ============================================================
# 4. Define conservative deterministic Branch C normalisation
# ============================================================
#
# Allowed:
# - Unicode NFKC;
# - Unicode-space standardisation;
# - typographic apostrophe standardisation;
# - dash/minus-glyph standardisation;
# - soft-hyphen removal;
# - CRLF/CR -> LF;
# - repeated horizontal whitespace collapsed;
# - trailing whitespace removed;
# - excessive blank-line runs standardised.
#
# Not applied:
# - a second structural conversion;
# - OCR;
# - page filtering/removal;
# - questionnaire-code rewriting;
# - semantic rewriting;
# - source spelling repair;
# - unit conversion;
# - numeric calculation;
# - manual reconstruction/correction;
# - reference-guided repair.
# ============================================================

UNICODE_SPACE_CHARACTERS = [
    "\u00a0", "\u1680", "\u2000", "\u2001", "\u2002",
    "\u2003", "\u2004", "\u2005", "\u2006", "\u2007",
    "\u2008", "\u2009", "\u200a", "\u202f", "\u205f",
    "\u3000"
]

APOSTROPHE_REPLACEMENTS = {
    "’": "'",
    "‘": "'",
    "‛": "'",
    "´": "'",
    "`": "'"
}

DASH_REPLACEMENTS = {
    "‐": "-",
    "‑": "-",
    "‒": "-",
    "–": "-",
    "—": "-",
    "−": "-"
}


def normalise_text_representation(text):
    text = unicodedata.normalize(
        "NFKC",
        str(text)
    )

    for character in UNICODE_SPACE_CHARACTERS:
        text = text.replace(
            character,
            " "
        )

    for source, target in APOSTROPHE_REPLACEMENTS.items():
        text = text.replace(
            source,
            target
        )

    for source, target in DASH_REPLACEMENTS.items():
        text = text.replace(
            source,
            target
        )

    text = text.replace(
        "\u00ad",
        ""
    )

    text = (
        text
        .replace("\r\n", "\n")
        .replace("\r", "\n")
    )

    normalised_lines = []

    for line in text.splitlines():
        line = re.sub(
            r"[ \t\f\v]+",
            " ",
            line
        ).rstrip()

        normalised_lines.append(
            line
        )

    text = "\n".join(
        normalised_lines
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return (
        text.strip()
        + "\n"
    )


In [6]:
# ============================================================
# 5. Apply Branch C normalisation to the COMPLETE frozen B representation
# ============================================================

NORMALISED_MARKDOWN = normalise_text_representation(
    SOURCE_B_MARKDOWN
)

if not NORMALISED_MARKDOWN.strip():
    raise ValueError(
        "D10 Branch C normalisation produced an empty representation."
    )

print(
    "Branch B characters:",
    len(SOURCE_B_MARKDOWN)
)

print(
    "Branch C characters:",
    len(NORMALISED_MARKDOWN)
)

print(
    "Representation changed:",
    SOURCE_B_MARKDOWN != NORMALISED_MARKDOWN
)


Branch B characters: 12708
Branch C characters: 12700
Representation changed: True


In [7]:
# ============================================================
# 6. Verify Branch C normalisation integrity
# ============================================================
#
# Transformation-aware checks are used because Unicode and whitespace
# normalisation are the intended C intervention.
# ============================================================

# ------------------------------------------------------------
# A. Page sequence
# ------------------------------------------------------------

parent_pages = PAGE_PATTERN.findall(
    SOURCE_B_MARKDOWN
)

branch_c_pages = PAGE_PATTERN.findall(
    NORMALISED_MARKDOWN
)

page_sequence_preserved = (
    parent_pages
    == branch_c_pages
    == expected_page_markers
)


# ------------------------------------------------------------
# B. Exact deterministic transformation reproducibility
# ------------------------------------------------------------

EXPECTED_NORMALISED_MARKDOWN = normalise_text_representation(
    SOURCE_B_MARKDOWN
)

deterministic_representation_verified = (
    NORMALISED_MARKDOWN
    == EXPECTED_NORMALISED_MARKDOWN
)


# ------------------------------------------------------------
# C. Critical source markers and questionnaire codes
# ------------------------------------------------------------

canonical_c = NORMALISED_MARKDOWN.casefold()

branch_c_marker_checks = {
    key:
        normalise_text_representation(
            marker
        ).strip().casefold()
        in canonical_c
    for key, marker
    in CRITICAL_PARENT_MARKERS.items()
}

all_critical_markers_preserved = all(
    branch_c_marker_checks.values()
)


observed_branch_c_codes = sorted(
    set(
        re.findall(
            r"\bBC\d{3}\b",
            NORMALISED_MARKDOWN
        )
    )
)

questionnaire_codes_preserved = (
    set(observed_branch_c_codes)
    == EXPECTED_QUESTIONNAIRE_CODES
)


# ------------------------------------------------------------
# D. Repeated UAE blocks and sample product rows remain represented
# ------------------------------------------------------------

branch_c_sample_product_checks = {
    label:
        label in NORMALISED_MARKDOWN
    for label in [
        "Produto a",
        "Produto b",
        "Produto c",
        "Produto d"
    ]
}

sample_product_rows_preserved = all(
    branch_c_sample_product_checks.values()
)


repeated_uae_status_count_c = len(
    re.findall(
        r"Situação da UAE perante a atividade",
        NORMALISED_MARKDOWN,
        flags=re.IGNORECASE
    )
)

repeated_uae_content_preserved = (
    repeated_uae_status_count_c
    == repeated_uae_status_count_parent
    and repeated_uae_status_count_c >= 3
)


# ------------------------------------------------------------
# E. Transformation-aware code/number preservation
# ------------------------------------------------------------

TOKEN_PATTERNS = {
    "questionnaire_codes":
        r"\bBC\d{3}\b",

    "dates":
        r"\b\d{4}/\d{2}/\d{2}\b",

    "postal_codes":
        r"\b\d{4}-\d{3}\b",

    "integers_and_decimals":
        r"(?<![\w])\d+(?:[.,]\d+)?(?![\w])"
}


def canonicalise_token(token):
    return (
        normalise_text_representation(
            token
        )
        .strip()
        .replace(" ", "")
        .casefold()
    )


token_preservation = {}

for label, pattern in TOKEN_PATTERNS.items():

    before = [
        canonicalise_token(token)
        for token in re.findall(
            pattern,
            SOURCE_B_MARKDOWN,
            flags=re.IGNORECASE
        )
    ]

    after = [
        canonicalise_token(token)
        for token in re.findall(
            pattern,
            NORMALISED_MARKDOWN,
            flags=re.IGNORECASE
        )
    ]

    before_counter = Counter(before)
    after_counter = Counter(after)

    missing = list(
        (
            before_counter
            - after_counter
        ).elements()
    )

    added = list(
        (
            after_counter
            - before_counter
        ).elements()
    )

    token_preservation[label] = {
        "count_before": len(before),
        "count_after": len(after),
        "missing_token_count": len(missing),
        "added_token_count": len(added),
        "passed": (
            len(missing) == 0
            and len(added) == 0
        )
    }


tokens_preserved = all(
    result["passed"]
    for result
    in token_preservation.values()
)


# ------------------------------------------------------------
# F. Final integrity decision
# ------------------------------------------------------------

normalisation_integrity_passed = bool(
    PARENT_EQUIVALENCE_PASSED
    and page_sequence_preserved
    and deterministic_representation_verified
    and all_critical_markers_preserved
    and questionnaire_codes_preserved
    and sample_product_rows_preserved
    and repeated_uae_content_preserved
    and tokens_preserved
)


normalisation_check = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "parent_branch":
        PARENT_BRANCH,

    "parent_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "parent_page_markers":
        parent_pages,

    "branch_C_page_markers":
        branch_c_pages,

    "page_sequence_preserved":
        page_sequence_preserved,

    "deterministic_representation_verified":
        deterministic_representation_verified,

    "critical_marker_checks":
        branch_c_marker_checks,

    "all_critical_markers_preserved":
        all_critical_markers_preserved,

    "observed_questionnaire_codes":
        observed_branch_c_codes,

    "questionnaire_codes_preserved":
        questionnaire_codes_preserved,

    "sample_product_row_checks":
        branch_c_sample_product_checks,

    "sample_product_rows_preserved_in_representation":
        sample_product_rows_preserved,

    "parent_repeated_uae_status_count":
        repeated_uae_status_count_parent,

    "branch_C_repeated_uae_status_count":
        repeated_uae_status_count_c,

    "repeated_uae_content_preserved":
        repeated_uae_content_preserved,

    "token_preservation":
        token_preservation,

    "tokens_preserved":
        tokens_preserved,

    "complete_4_page_representation_retained":
        True,

    "source_scope_filtering_applied":
        False,

    "page_removal_applied":
        False,

    "page_cropping_applied":
        False,

    "branch_B_structural_conversion_inherited":
        True,

    "branch_B_regeneration_attempted":
        False,

    "ocr_applied":
        False,

    "unicode_nfkc_normalisation_applied":
        True,

    "unicode_space_standardisation_applied":
        True,

    "apostrophe_standardisation_applied":
        True,

    "dash_and_minus_standardisation_applied":
        True,

    "soft_hyphen_removal_applied":
        True,

    "line_endings_standardised":
        True,

    "horizontal_whitespace_normalisation_applied":
        True,

    "semantic_harmonisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "questionnaire_code_rewriting_applied":
        False,

    "source_spelling_correction_applied":
        False,

    "unit_conversion_applied":
        False,

    "numeric_calculation_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "reference_values_used_for_transformation":
        False,

    "normalisation_integrity_passed":
        normalisation_integrity_passed
}


NORMALISATION_CHECK_PATH.write_text(
    json.dumps(
        normalisation_check,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        normalisation_check,
        ensure_ascii=False,
        indent=2
    )
)


if not normalisation_integrity_passed:
    raise ValueError(
        "D10 Branch C normalisation-integrity checks failed. "
        "Inspect parent equivalence, page/marker/code preservation, "
        "UAE/sample-row preservation and transformation-aware tokens."
    )


{
  "document_id": "D10",
  "branch": "C",
  "parent_branch": "B",
  "parent_equivalence_passed": true,
  "parent_page_markers": [
    "1",
    "2",
    "3",
    "4"
  ],
  "branch_C_page_markers": [
    "1",
    "2",
    "3",
    "4"
  ],
  "page_sequence_preserved": true,
  "deterministic_representation_verified": true,
  "critical_marker_checks": {
    "survey_title": true,
    "registration_number": true,
    "validity_date": true,
    "section_i": true,
    "section_ii": true,
    "section_iii": true,
    "section_iv": true,
    "uae": true,
    "quantities_produced": true,
    "quantities_sold": true,
    "sales_value": true,
    "filling_instructions": true,
    "prodcom": true
  },
  "all_critical_markers_preserved": true,
  "observed_questionnaire_codes": [
    "BC001",
    "BC005",
    "BC007",
    "BC010",
    "BC015",
    "BC020",
    "BC025",
    "BC030"
  ],
  "questionnaire_codes_preserved": true,
  "sample_product_row_checks": {
    "Produto a": true,
    "Produto b": t

In [8]:
# ============================================================
# 7. Save Branch C normalised representation
# ============================================================

REPRESENTATION_PATH.write_text(
    NORMALISED_MARKDOWN,
    encoding="utf-8"
)

REPRESENTATION_SHA256 = sha256_file(
    REPRESENTATION_PATH
)

print(
    "Saved:",
    REPRESENTATION_PATH.name
)

print(
    "Representation SHA-256:",
    REPRESENTATION_SHA256
)


Saved: D10_branch_C_normalised_markdown.md
Representation SHA-256: 6fef221e9ac214f34ac330a7a63967290edeaa56e6f2f18e10b3cb0bd5ccc651


In [9]:
# ============================================================
# 8. Create controlled Branch C extraction prompt
# ============================================================
#
# Substantive extraction task/schema are frozen from final Branch B.
# Expected record/category counts are deliberately NOT disclosed.
# ============================================================

BRANCH_C_PROMPT = 'You are an information extraction assistant.\n\nExtract the predefined structural and semantic questionnaire records\nrepresented within the defined scope of the attached deterministically\nnormalised structural Markdown representation of:\n\nIMPI — Inquérito Mensal à Produção Industrial.\n\nTreat the attached deterministically normalised structural Markdown\nrepresentation as the only source of information.\n\nThe supplied representation corresponds to the complete four physical\npages of the original questionnaire PDF.\n\nThe questionnaire is blank. Extract represented questionnaire\nstructure, labels, metadata, reusable template elements, table fields\nand instructions. Do not invent or infer respondent answers.\n\nFor every included record return exactly these fields:\n\n- Category\n- Section\n- Field or Concept\n- Description\n- Code\n- Expected Value Type\n- Source Location\n\nUse exactly one of these Category values:\n\n- Instrument metadata\n- Questionnaire field\n- UAE template element\n- Product table field\n- Instruction\n\n\n1. Instrument metadata\n\nFrom the header, legal notice and response-contact areas represented\nfrom physical PDF page 1, extract one record for each of these\npredefined concepts:\n\n- Survey name\n- Statistical system\n- Legal basis\n- INE registration number\n- Validity date\n- Electronic response URL\n- Contact email\n- Contact telephone\n\nRead their represented content directly from the source representation.\n\nDo not create separate records from decorative branding or graphical\nelements.\n\n\n2. Questionnaire fields\n\nExtract the predefined response fields represented from physical PDF\npage 1.\n\nReference-data area:\n- Referência dos dados\n- NIF\n\nSection I — Identificação da unidade estatística:\n- Número de identificação fiscal (NIF)\n- Homepage\n- Designação social\n- Distrito/Ilha\n- Município\n- Freguesia\n- Endereço\n- Localidade\n- Código postal\n- Telefone\n- Fax\n- e-mail\n\nSection II — Situação da unidade estatística no período de referência\ndos dados:\n- Situação na atividade\n- Aguarda início de atividade\n- Em atividade\n- Atividade suspensa em\n- Atividade cessada em\n- Nº dias de atividade no período de referência\n- Atividade económica principal (CAE Rev. 3)\n- Ocorreu algum facto relevante no período de referência dos dados?\n- Indique qual\n- Data\n\nSection III — Observações:\n- Observações\n\nSection IV — Responsável pelo preenchimento:\n- Nome contacto\n- Telefone\n- Fax\n- e-mail\n- Função\n- Assinatura\n- Data\n\nBlank boxes, blank lines and empty response areas represent fields.\nDo not treat them as missing respondent observations.\n\nWhere a printed questionnaire code is represented as associated with a\ntarget field, preserve that code exactly in Code.\n\nUse null for Code when no printed questionnaire code is represented.\n\nDo not infer a code from a neighbouring field or from external\nknowledge.\n\n\n3. UAE template elements\n\nPhysical PDF page 2 contains repeated Unidade de Atividade Económica\n(UAE) blocks.\n\nTreat the repeated blocks as multiple source instances of one reusable\ntemplate.\n\nExtract each distinct reusable template element once:\n\n- Código da UAE\n- Designação da UAE\n- Situação da UAE perante a atividade\n- Observações da UAE\n- Confirmar\n- Produtos\n\nDo not create duplicate records merely because the same UAE template is\nrepresented repeatedly.\n\nClassify these records as:\n\nCategory = "UAE template element"\nSection = "UAE information"\n\n\n4. Product table fields\n\nFrom the product-entry table represented from physical PDF page 3,\nextract one record for each of these structural fields:\n\n- NIF\n- UAE\n- Período de Referência\n- Nº\n- Produto\n- Unid.\n- Código\n- Quantidades produzidas\n- Quantidades vendidas\n- Valor das vendas / prestação de serviços\n- Observações empresa\n- Observações INE\n\nPreserve the represented table-column relationships.\n\nThe rows labelled Produto a, Produto b, Produto c and Produto d are\nsample/template examples. Do not extract them as respondent\nobservations or additional product records.\n\nDo not extract their sample product codes or sample unit letters as\nseparate observations.\n\n\n5. Instructions and explanatory definitions\n\nExtract the principal predefined instructional and explanatory concepts\nrepresented from physical PDF pages 2 and 4.\n\nFrom the UAE information on page 2:\n- Unidade de Atividade Económica (UAE)\n\nFrom the filling instructions on page 4:\n- Questionnaire scope\n- Unidade monetária\n- Arredondamentos\n- Exemplo de arredondamento\n\nFrom the explanatory notes on page 4:\n- Empresa\n- Unidade de Atividade Económica (UAE)\n- Produtos\n- Quantidades produzidas\n- Quantidades vendidas\n- Valor das vendas / prestação de serviços\n\nFor each instruction or definition, provide a concise source-grounded\nDescription preserving the substantive represented rule or definition.\n\nDo not split supporting sentences into additional records outside this\npredefined scope.\n\n\nField rules:\n\nCategory:\n- Use exactly one of the five Category labels defined above.\n\nSection:\n- Identify the source section or structural region containing the\n  represented element.\n- Use concise stable section labels.\n\nField or Concept:\n- Use the predefined field or concept label corresponding to the item\n  being extracted.\n- Preserve Portuguese source wording where the item is a visible source\n  label.\n\nDescription:\n- Provide a concise source-grounded description of the represented\n  field, metadata item, template element, table field or instruction.\n- Do not add external interpretation.\n- For instructions and definitions, retain the substantive meaning\n  explicitly represented by the source.\n\nCode:\n- Preserve a printed questionnaire code only when represented as\n  associated with the target element.\n- Use null when no printed code applies.\n- Do not infer or transfer codes between neighbouring elements.\n\nExpected Value Type:\n- Assign a concise structural expected-value type based only on the\n  represented questionnaire element or semantic role of the target item.\n- Use consistent labels such as:\n  text\n  identifier\n  numeric identifier\n  date\n  URL\n  email\n  telephone number\n  reference period\n  postal code\n  category\n  checkbox\n  integer\n  CAE code\n  yes or no\n  free text\n  signature\n  button or action\n  month or period\n  unit\n  product code\n  numeric quantity\n  monetary value\n  instruction\n  definition\n  example\n- Do not infer actual respondent values.\n\nSource Location:\n- Use physical PDF page references grounded in the Markdown page\n  boundaries.\n- Include a concise source region or section.\n- Examples include:\n  "PDF page 1 — Header"\n  "PDF page 1 — Reference data"\n  "PDF page 1 — Section I"\n  "PDF page 1 — Section II"\n  "PDF page 1 — Section III"\n  "PDF page 1 — Section IV"\n  "PDF page 2 — UAE information"\n  "PDF page 3 — Product production table"\n  "PDF page 4 — Filling instructions"\n  "PDF page 4 — Explanatory notes"\n\n\nAdditional extraction rules:\n\n- Use only information explicitly represented in the attached\n  deterministically normalised structural Markdown.\n- Preserve Portuguese wording, labels and printed codes where relevant.\n- Do not invent respondent answers from blank response fields.\n- Do not treat blank response areas as null observations.\n- Do not duplicate repeated UAE template elements.\n- Do not treat sample product rows as respondent observations.\n- Do not calculate, infer, derive, repair or introduce information.\n- Do not use external knowledge.\n- Do not follow external links.\n- Do not reconstruct hidden form data.\n- Do not create records outside the predefined extraction scope.\n- Ignore synthetic Markdown block labels and bounding-box metadata\n  except as structural cues.\n- Verify that every item within the defined source scope has been\n  processed.\n- Return only valid JSON.\n- Do not include Markdown fences, explanations or commentary.\n- Keep the exact field names and field order defined below.\n\nExpected JSON structure:\n\n{\n  "document_id": "D10",\n  "branch": "C",\n  "records": [\n    {\n      "Category": null,\n      "Section": null,\n      "Field or Concept": null,\n      "Description": null,\n      "Code": null,\n      "Expected Value Type": null,\n      "Source Location": null\n    }\n  ]\n}\n\nReturn only the JSON object.'

PROMPT_PATH.write_text(
    BRANCH_C_PROMPT,
    encoding="utf-8"
)

PROMPT_SHA256 = sha256_file(
    PROMPT_PATH
)

print(
    "Prompt saved:",
    PROMPT_PATH.name
)

print(
    "Prompt SHA-256:",
    PROMPT_SHA256
)


Prompt saved: D10_branch_C_prompt.txt
Prompt SHA-256: deff11f7ccda8c6fb4ab5d0210f0bc2a0ba546410ab77923cb350f7befb7a4b6


In [10]:
# ============================================================
# 9. Create representation and pre-extraction metadata
# ============================================================

REPRESENTATION_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "parent_branch":
        PARENT_BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "parent_B_representation_file":
        BRANCH_B_REPRESENTATION_PATH.name,

    "parent_B_representation_sha256":
        UPLOADED_BRANCH_B_SHA256,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "parent_B_equivalence_method":
        "Frozen Branch B representation SHA-256 verification",

    "representation_type":
        (
            "Complete frozen Branch B page-aware layout-aware structural "
            "Markdown with deterministic normalisation"
        ),

    "representation_file":
        REPRESENTATION_PATH.name,

    "representation_sha256":
        REPRESENTATION_SHA256,

    "complete_4_page_representation_retained":
        True,

    "scope_enforced_by_prompt_not_representation_filtering":
        True,

    "structural_conversion_inherited_from_branch_B":
        True,

    "branch_B_regeneration_attempted":
        False,

    "repeated_uae_blocks_retained":
        True,

    "sample_product_rows_retained":
        True,

    "normalisation_applied":
        True,

    "semantic_rewriting_applied":
        False,

    "questionnaire_code_rewriting_applied":
        False,

    "unit_conversion_applied":
        False,

    "numeric_calculation_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "reference_values_used_for_transformation":
        False,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ]
}


REPRESENTATION_METADATA_PATH.write_text(
    json.dumps(
        REPRESENTATION_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


EXPERIMENT_METADATA_PRE = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "parent_branch":
        PARENT_BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        SOURCE_HASH_MATCH,

    "input_representation":
        "Complete deterministically normalised layout-aware structural Markdown",

    "representation_file":
        REPRESENTATION_PATH.name,

    "representation_sha256":
        REPRESENTATION_SHA256,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ],

    "direct_document_ingestion":
        False,

    "structural_conversion_applied":
        True,

    "structural_conversion_inherited_from_branch_B":
        True,

    "normalisation_applied":
        True,

    "complete_source_document_retained":
        True,

    "source_scope_filtering_applied":
        False,

    "reference_values_disclosed_to_model":
        False,

    "reference_values_used_for_transformation":
        False,

    "expected_record_count_disclosed_to_model":
        False,

    "expected_category_counts_disclosed_to_model":
        False,

    "manual_response_repair_permitted":
        False,

    "expected_output_format":
        "JSON object",

    "prompt_file":
        PROMPT_PATH.name,

    "prompt_sha256":
        PROMPT_SHA256,

    "execution_environment":
        "Independent ChatGPT conversation",

    "model":
        "GPT-5.5",

    "created_at":
        datetime.now().isoformat(),

    "python_version":
        sys.version,

    "platform":
        platform.platform(),

    "validation_status":
        "Pending independent Branch C extraction and Stage 4 Validation C"
}


EXPERIMENT_METADATA_PRE_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA_PRE,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        EXPERIMENT_METADATA_PRE,
        ensure_ascii=False,
        indent=2
    )
)


{
  "document_id": "D10",
  "document_name": "IMPI — Inquérito Mensal à Produção Industrial",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "source_file": "D10 - IMPI_QUESTIONARIO INE Portugal 2026.pdf",
  "source_sha256": "fb72aac548f61578bc9ba52448793b81bf61e37eff146f9519124d0794a4f4e5",
  "source_verified": true,
  "input_representation": "Complete deterministically normalised layout-aware structural Markdown",
  "representation_file": "D10_branch_C_normalised_markdown.md",
  "representation_sha256": "6fef221e9ac214f34ac330a7a63967290edeaa56e6f2f18e10b3cb0bd5ccc651",
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "direct_document_ingestion": false,
  "structural_conversion_applied": true,
  "structural_conversion_inherited_from_branch_B": true,
  "normalisation_applied": true,
  "complete_source_document_retained": true,
  "source_scope_filtering_applied": false,
  "reference_values_disclosed_to_model":

In [11]:
# ============================================================
# 10. Final pre-extraction control check
# ============================================================

PRECHECK = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "source_identity_verified":
        SOURCE_HASH_MATCH,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ],

    "complete_4_page_representation_retained":
        True,

    "page_sequence_preserved":
        page_sequence_preserved,

    "all_critical_markers_preserved":
        all_critical_markers_preserved,

    "questionnaire_codes_preserved":
        questionnaire_codes_preserved,

    "repeated_uae_content_preserved":
        repeated_uae_content_preserved,

    "sample_product_rows_preserved_in_representation":
        sample_product_rows_preserved,

    "representation_exists":
        REPRESENTATION_PATH.exists(),

    "prompt_exists":
        PROMPT_PATH.exists(),

    "expected_record_count_disclosed_to_model":
        False,

    "expected_category_counts_disclosed_to_model":
        False,

    "reference_values_used_for_transformation":
        False,

    "ready_for_independent_llm_execution":
        bool(
            SOURCE_HASH_MATCH
            and PARENT_EQUIVALENCE_PASSED
            and normalisation_check[
                "normalisation_integrity_passed"
            ]
            and REPRESENTATION_PATH.exists()
            and PROMPT_PATH.exists()
        )
}


PRECHECK_PATH.write_text(
    json.dumps(
        PRECHECK,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        PRECHECK,
        ensure_ascii=False,
        indent=2
    )
)


if not PRECHECK[
    "ready_for_independent_llm_execution"
]:
    raise ValueError(
        "D10 Branch C is not ready for independent LLM execution."
    )


{
  "document_id": "D10",
  "branch": "C",
  "source_identity_verified": true,
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "complete_4_page_representation_retained": true,
  "page_sequence_preserved": true,
  "all_critical_markers_preserved": true,
  "questionnaire_codes_preserved": true,
  "repeated_uae_content_preserved": true,
  "sample_product_rows_preserved_in_representation": true,
  "representation_exists": true,
  "prompt_exists": true,
  "expected_record_count_disclosed_to_model": false,
  "expected_category_counts_disclosed_to_model": false,
  "reference_values_used_for_transformation": false,
  "ready_for_independent_llm_execution": true
}


In [12]:
# ============================================================
# 11. Download pre-extraction Branch C artefacts
# ============================================================

for path in [
    PARENT_CHECK_PATH,
    NORMALISATION_CHECK_PATH,
    REPRESENTATION_PATH,
    REPRESENTATION_METADATA_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH,
    PRECHECK_PATH
]:
    files.download(
        path
    )


print(
    "\nIndependent extraction instructions:\n"
    "1. Open a new independent ChatGPT conversation.\n"
    "2. Upload ONLY D10_branch_C_normalised_markdown.md.\n"
    "3. Submit D10_branch_C_prompt.txt exactly once.\n"
    "4. Do not upload the original PDF, Branch B artefacts, Stage 1 "
    "reference values, or previous extraction outputs.\n"
    "5. Do not manually repair, correct, or regenerate the response.\n"
    "6. Save the complete response exactly as returned in a plain-text file."
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Independent extraction instructions:
1. Open a new independent ChatGPT conversation.
2. Upload ONLY D10_branch_C_normalised_markdown.md.
3. Submit D10_branch_C_prompt.txt exactly once.
4. Do not upload the original PDF, Branch B artefacts, Stage 1 reference values, or previous extraction outputs.
5. Do not manually repair, correct, or regenerate the response.
6. Save the complete response exactly as returned in a plain-text file.


In [13]:
# ============================================================
# 12. Upload and preserve the complete raw Branch C response
# ============================================================

uploaded_response = files.upload()

if len(uploaded_response) != 1:
    raise ValueError(
        "Upload exactly one complete raw D10 Branch C response file."
    )


RAW_RESPONSE_SOURCE = Path(
    next(
        iter(
            uploaded_response
        )
    )
)


RAW_RESPONSE_TEXT = (
    RAW_RESPONSE_SOURCE.read_text(
        encoding="utf-8"
    )
)


RAW_RESPONSE_PATH.write_text(
    RAW_RESPONSE_TEXT,
    encoding="utf-8"
)


RAW_RESPONSE_SHA256 = sha256_file(
    RAW_RESPONSE_PATH
)


print(
    "Raw response preserved unchanged."
)

print(
    "Raw response SHA-256:",
    RAW_RESPONSE_SHA256
)


Saving D10_branch_C_raw_response.txt to D10_branch_C_raw_response.txt
Raw response preserved unchanged.
Raw response SHA-256: 224fc5bf11f9cbc983d07bd6e08946f47151d42b7b878c12e2a14911943bdd8a


In [14]:
# ============================================================
# 13. Parse raw response WITHOUT repair
# ============================================================

valid_json = True
json_parsing_error = None
parsed_response = None


try:
    parsed_response = json.loads(
        RAW_RESPONSE_TEXT
    )

except json.JSONDecodeError as exc:
    valid_json = False
    json_parsing_error = str(exc)


top_level_object_valid = (
    valid_json
    and isinstance(
        parsed_response,
        dict
    )
)

document_id_present = (
    top_level_object_valid
    and "document_id" in parsed_response
)

document_id_correct = (
    document_id_present
    and parsed_response.get(
        "document_id"
    )
    == DOCUMENT_ID
)

branch_present = (
    top_level_object_valid
    and "branch" in parsed_response
)

branch_correct = (
    branch_present
    and parsed_response.get(
        "branch"
    )
    == BRANCH
)

records_present = (
    top_level_object_valid
    and "records" in parsed_response
)

records_is_list = (
    records_present
    and isinstance(
        parsed_response.get(
            "records"
        ),
        list
    )
)

records_evaluable = bool(
    valid_json
    and top_level_object_valid
    and document_id_correct
    and branch_correct
    and records_is_list
)

extracted_records = (
    parsed_response["records"]
    if records_evaluable
    else []
)

observed_record_count = (
    len(
        extracted_records
    )
    if records_evaluable
    else None
)


print("Valid JSON:", valid_json)
print("Records evaluable:", records_evaluable)
print("Observed records:", observed_record_count)

if json_parsing_error:
    print(
        "JSON parsing error:",
        json_parsing_error
    )


Valid JSON: True
Records evaluable: True
Observed records: 69


In [15]:
# ============================================================
# 14. Validate record schema and field types
# ============================================================

record_structure_issues = []
field_type_issues = []
missing_mandatory_values = []


if records_evaluable:

    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(
            record,
            dict
        ):
            record_structure_issues.append({
                "record_index":
                    record_index,

                "issue":
                    "Record is not a JSON object"
            })

            continue


        observed_fields = list(
            record.keys()
        )

        if observed_fields != EXPECTED_FIELDS:
            record_structure_issues.append({
                "record_index":
                    record_index,

                "issue":
                    "Field names or field order differ",

                "expected_fields":
                    EXPECTED_FIELDS,

                "observed_fields":
                    observed_fields
            })


        for field in EXPECTED_FIELDS:

            value = record.get(
                field
            )

            if field == "Code":

                if (
                    value is not None
                    and not isinstance(
                        value,
                        str
                    )
                ):
                    field_type_issues.append({
                        "record_index":
                            record_index,

                        "field":
                            field,

                        "observed_type":
                            type(
                                value
                            ).__name__
                    })

            else:

                if not isinstance(
                    value,
                    str
                ):
                    field_type_issues.append({
                        "record_index":
                            record_index,

                        "field":
                            field,

                        "observed_type":
                            type(
                                value
                            ).__name__
                    })

                elif not value.strip():
                    missing_mandatory_values.append({
                        "record_index":
                            record_index,

                        "field":
                            field
                    })


record_schema_valid = (
    len(
        record_structure_issues
    )
    == 0
    if records_evaluable
    else None
)

field_types_valid = (
    len(
        field_type_issues
    )
    == 0
    if records_evaluable
    else None
)

mandatory_fields_complete = (
    len(
        missing_mandatory_values
    )
    == 0
    if records_evaluable
    else None
)


print(
    "Record schema valid:",
    record_schema_valid
)

print(
    "Field types valid:",
    field_types_valid
)

print(
    "Mandatory fields complete:",
    mandatory_fields_complete
)

print(
    "Structure issues:",
    len(
        record_structure_issues
    )
)

print(
    "Type issues:",
    len(
        field_type_issues
    )
)


Record schema valid: True
Field types valid: True
Mandatory fields complete: True
Structure issues: 0
Type issues: 0


In [16]:
# ============================================================
# 15. Content/scope diagnostics kept separate from schema validity
# ============================================================

if records_evaluable:

    record_count_valid = (
        observed_record_count
        == EXPECTED_RECORD_COUNT
    )


    observed_category_counts = dict(
        Counter(
            record.get(
                "Category"
            )
            for record
            in extracted_records
            if isinstance(
                record,
                dict
            )
        )
    )


    categories_valid = set(
        observed_category_counts
    ).issubset(
        ALLOWED_CATEGORIES
    )


    category_counts_valid = (
        observed_category_counts
        == EXPECTED_CATEGORY_COUNTS
    )


    duplicate_counter = Counter(
        tuple(
            json.dumps(
                record.get(field),
                ensure_ascii=False,
                sort_keys=True
            )
            for field in EXPECTED_FIELDS
        )
        for record in extracted_records
        if isinstance(
            record,
            dict
        )
    )


    duplicate_complete_records = [
        {
            "record":
                list(
                    key
                ),

            "occurrence_count":
                count
        }
        for key, count
        in duplicate_counter.items()
        if count > 1
    ]


    duplicate_complete_record_count = len(
        duplicate_complete_records
    )


    observed_non_null_codes = sorted({
        record.get("Code")
        for record in extracted_records
        if (
            isinstance(record, dict)
            and isinstance(
                record.get("Code"),
                str
            )
            and record.get("Code").strip()
        )
    })


    questionnaire_code_set_valid = (
        set(
            observed_non_null_codes
        )
        == EXPECTED_QUESTIONNAIRE_CODES
    )


    uae_template_records = [
        record
        for record
        in extracted_records
        if (
            isinstance(record, dict)
            and record.get("Category")
            == "UAE template element"
        )
    ]


    observed_uae_template_labels = {
        record.get(
            "Field or Concept"
        )
        for record
        in uae_template_records
    }


    uae_template_complete = (
        observed_uae_template_labels
        == EXPECTED_UAE_TEMPLATE_LABELS
    )


    uae_template_counts = Counter(
        record.get(
            "Field or Concept"
        )
        for record
        in uae_template_records
    )


    uae_template_not_duplicated = (
        all(
            count == 1
            for count
            in uae_template_counts.values()
        )
        and len(
            uae_template_records
        )
        == len(
            EXPECTED_UAE_TEMPLATE_LABELS
        )
    )


    product_table_records = [
        record
        for record
        in extracted_records
        if (
            isinstance(record, dict)
            and record.get("Category")
            == "Product table field"
        )
    ]


    observed_product_table_fields = {
        record.get(
            "Field or Concept"
        )
        for record
        in product_table_records
    }


    product_table_scope_complete = (
        observed_product_table_fields
        == EXPECTED_PRODUCT_TABLE_FIELDS
    )


    extraction_search_text = json.dumps(
        extracted_records,
        ensure_ascii=False
    ).casefold()


    sample_product_rows_excluded = not any(
        marker
        in extraction_search_text
        for marker
        in [
            "produto a",
            "produto b",
            "produto c",
            "produto d",
            "111111111111"
        ]
    )


    reference_period_field_present = any(
        (
            isinstance(record, dict)
            and record.get("Category")
            == "Questionnaire field"
            and record.get("Field or Concept")
            == "Referência dos dados"
        )
        for record
        in extracted_records
    )


else:

    record_count_valid = None
    observed_category_counts = None
    categories_valid = None
    category_counts_valid = None
    duplicate_complete_records = None
    duplicate_complete_record_count = None
    observed_non_null_codes = None
    questionnaire_code_set_valid = None
    observed_uae_template_labels = None
    uae_template_complete = None
    uae_template_counts = None
    uae_template_not_duplicated = None
    observed_product_table_fields = None
    product_table_scope_complete = None
    sample_product_rows_excluded = None
    reference_period_field_present = None


scope_complete = bool(
    record_count_valid
    and category_counts_valid
    and uae_template_complete
    and uae_template_not_duplicated
    and product_table_scope_complete
    and sample_product_rows_excluded
    and reference_period_field_present
) if records_evaluable else False


CONTENT_DIAGNOSTICS = {
    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches_reference":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "categories_valid":
        categories_valid,

    "category_counts_match_reference":
        category_counts_valid,

    "mandatory_fields_complete":
        mandatory_fields_complete,

    "missing_mandatory_value_count":
        (
            len(
                missing_mandatory_values
            )
            if records_evaluable
            else None
        ),

    "duplicate_complete_record_count":
        duplicate_complete_record_count,

    "duplicate_complete_records":
        duplicate_complete_records,

    "duplicate_check_is_diagnostic_only":
        True,

    "expected_questionnaire_codes":
        sorted(
            EXPECTED_QUESTIONNAIRE_CODES
        ),

    "observed_non_null_codes":
        observed_non_null_codes,

    "questionnaire_code_set_valid":
        questionnaire_code_set_valid,

    "expected_uae_template_labels":
        sorted(
            EXPECTED_UAE_TEMPLATE_LABELS
        ),

    "observed_uae_template_labels":
        (
            sorted(
                observed_uae_template_labels
            )
            if observed_uae_template_labels
            is not None
            else None
        ),

    "uae_template_complete":
        uae_template_complete,

    "uae_template_counts":
        (
            dict(
                uae_template_counts
            )
            if uae_template_counts
            is not None
            else None
        ),

    "uae_template_not_duplicated":
        uae_template_not_duplicated,

    "expected_product_table_fields":
        sorted(
            EXPECTED_PRODUCT_TABLE_FIELDS
        ),

    "observed_product_table_fields":
        (
            sorted(
                observed_product_table_fields
            )
            if observed_product_table_fields
            is not None
            else None
        ),

    "product_table_scope_complete":
        product_table_scope_complete,

    "sample_product_rows_excluded":
        sample_product_rows_excluded,

    "reference_period_field_present":
        reference_period_field_present
}


print(
    json.dumps(
        CONTENT_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)


{
  "expected_record_count": 69,
  "observed_record_count": 69,
  "record_count_matches_reference": true,
  "expected_category_counts": {
    "Instrument metadata": 8,
    "Questionnaire field": 32,
    "UAE template element": 6,
    "Product table field": 12,
    "Instruction": 11
  },
  "observed_category_counts": {
    "Instrument metadata": 8,
    "Questionnaire field": 32,
    "UAE template element": 6,
    "Product table field": 12,
    "Instruction": 11
  },
  "categories_valid": true,
  "category_counts_match_reference": true,
  "mandatory_fields_complete": true,
  "missing_mandatory_value_count": 0,
  "duplicate_complete_record_count": 0,
  "duplicate_complete_records": [],
  "duplicate_check_is_diagnostic_only": true,
  "expected_questionnaire_codes": [
    "BC001",
    "BC005",
    "BC007",
    "BC010",
    "BC015",
    "BC020",
    "BC025",
    "BC030"
  ],
  "observed_non_null_codes": [
    "BC001",
    "BC005",
    "BC007",
    "BC010",
    "BC015",
    "BC020",
    "BC02

In [17]:
# ============================================================
# 16. Determine technical/schema validity
# ============================================================
#
# Expected count/category agreement, mandatory-content completeness,
# code-set agreement and D10 scope diagnostics are deliberately NOT
# conditions for technical/schema validity.
# ============================================================

structure_valid = bool(
    valid_json
    and top_level_object_valid
    and document_id_correct
    and branch_correct
    and records_is_list
    and record_schema_valid is True
    and field_types_valid is True
)


STRUCTURE_CHECK = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_representation":
        "Complete deterministically normalised layout-aware structural Markdown",

    "valid_json":
        bool(
            valid_json
        ),

    "json_parsing_error":
        json_parsing_error,

    "top_level_object_valid":
        bool(
            top_level_object_valid
        ),

    "document_id_correct":
        bool(
            document_id_correct
        ),

    "branch_correct":
        bool(
            branch_correct
        ),

    "records_is_list":
        bool(
            records_is_list
        ),

    "records_evaluable":
        bool(
            records_evaluable
        ),

    "record_schema_valid":
        record_schema_valid,

    "record_structure_issues":
        (
            record_structure_issues
            if records_evaluable
            else None
        ),

    "field_types_valid":
        field_types_valid,

    "field_type_issues":
        (
            field_type_issues
            if records_evaluable
            else None
        ),

    "content_diagnostics":
        CONTENT_DIAGNOSTICS,

    "structure_valid":
        bool(
            structure_valid
        ),

    "scope_complete":
        bool(
            scope_complete
        )
}


STRUCTURE_CHECK_PATH.write_text(
    json.dumps(
        STRUCTURE_CHECK,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        STRUCTURE_CHECK,
        ensure_ascii=False,
        indent=2
    )
)


{
  "document_id": "D10",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "input_representation": "Complete deterministically normalised layout-aware structural Markdown",
  "valid_json": true,
  "json_parsing_error": null,
  "top_level_object_valid": true,
  "document_id_correct": true,
  "branch_correct": true,
  "records_is_list": true,
  "records_evaluable": true,
  "record_schema_valid": true,
  "record_structure_issues": [],
  "field_types_valid": true,
  "field_type_issues": [],
  "content_diagnostics": {
    "expected_record_count": 69,
    "observed_record_count": 69,
    "record_count_matches_reference": true,
    "expected_category_counts": {
      "Instrument metadata": 8,
      "Questionnaire field": 32,
      "UAE template element": 6,
      "Product table field": 12,
      "Instruction": 11
    },
    "observed_category_counts": {
      "Instrument metadata": 8,
      "Questionnaire field": 32,
      "UAE template element": 6,
      "Product table fiel

In [18]:
# ============================================================
# 17. Preserve parsed extraction only when records are evaluable
# ============================================================

parsed_extraction_created = False
parsed_extraction_sha256 = None


if records_evaluable:

    canonical_extraction = {
        "document_id":
            DOCUMENT_ID,

        "branch":
            BRANCH,

        "records":
            extracted_records
    }


    PARSED_EXTRACTION_PATH.write_text(
        json.dumps(
            canonical_extraction,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )


    parsed_extraction_sha256 = sha256_file(
        PARSED_EXTRACTION_PATH
    )

    parsed_extraction_created = True


    print(
        "Parsed extraction saved:",
        PARSED_EXTRACTION_PATH.name
    )

else:

    print(
        "No parsed extraction created because the preserved raw response "
        "does not contain an evaluable JSON records structure."
    )


Parsed extraction saved: D10_branch_C_parsed_extraction.json


In [19]:
# ============================================================
# 18. Create final experiment metadata and summary
# ============================================================

EXPERIMENT_METADATA = {
    **EXPERIMENT_METADATA_PRE,

    "raw_response_file":
        RAW_RESPONSE_PATH.name,

    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "parsed_extraction_file":
        (
            PARSED_EXTRACTION_PATH.name
            if parsed_extraction_created
            else None
        ),

    "parsed_extraction_sha256":
        parsed_extraction_sha256,

    "json_valid":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "observed_record_count":
        observed_record_count,

    "observed_category_counts":
        observed_category_counts,

    "structure_check_file":
        STRUCTURE_CHECK_PATH.name,

    "structure_valid":
        bool(
            structure_valid
        ),

    "notes": (
        "Branch C applies deterministic non-semantic normalisation to the "
        "exact frozen Branch B page-aware layout-aware structural Markdown. "
        "The complete four-page representation, repeated UAE source blocks "
        "and sample product rows remain represented. Stage 1 expected counts "
        "and questionnaire-code expectations are not supplied to the model. "
        "Accuracy is evaluated separately in Validation C."
    )
}


EXPERIMENT_METADATA_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


EXPERIMENT_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "parent_branch":
        PARENT_BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        SOURCE_HASH_MATCH,

    "input_representation":
        "Complete deterministically normalised layout-aware structural Markdown",

    "representation_file":
        REPRESENTATION_PATH.name,

    "representation_sha256":
        REPRESENTATION_SHA256,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ],

    "structural_conversion_inherited_from_branch_B":
        True,

    "branch_B_regeneration_attempted":
        False,

    "normalisation_applied":
        True,

    "complete_source_document_retained":
        True,

    "source_scope_filtering_applied":
        False,

    "reference_values_used_for_transformation":
        False,

    "expected_record_count_disclosed_to_model":
        False,

    "expected_category_counts_disclosed_to_model":
        False,

    "raw_response_preserved":
        True,

    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "valid_json":
        bool(
            valid_json
        ),

    "records_evaluable":
        bool(
            records_evaluable
        ),

    "record_schema_valid":
        record_schema_valid,

    "field_types_valid":
        field_types_valid,

    "structure_valid":
        bool(
            structure_valid
        ),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_match":
        category_counts_valid,

    "scope_complete":
        bool(
            scope_complete
        ),

    "duplicate_complete_record_count":
        duplicate_complete_record_count,

    "questionnaire_code_set_valid":
        questionnaire_code_set_valid,

    "reference_period_field_present":
        reference_period_field_present,

    "uae_template_complete":
        uae_template_complete,

    "uae_template_not_duplicated":
        uae_template_not_duplicated,

    "product_table_scope_complete":
        product_table_scope_complete,

    "sample_product_rows_excluded":
        sample_product_rows_excluded,

    "parsed_extraction_created":
        parsed_extraction_created,

    "parsed_extraction_sha256":
        parsed_extraction_sha256,

    "accuracy_validation_completed":
        False,

    "validation_status":
        (
            "Pending Stage 4 Branch C validation against the fixed Stage 1 "
            "reference dataset using Branch A-frozen D10 comparison rules"
            if records_evaluable
            else
            "Not content-evaluable because the preserved Branch C response "
            "does not contain an evaluable JSON records structure"
        )
}


EXPERIMENT_SUMMARY_PATH.write_text(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    )
)


{
  "document_id": "D10",
  "document_name": "IMPI — Inquérito Mensal à Produção Industrial",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "source_file": "D10 - IMPI_QUESTIONARIO INE Portugal 2026.pdf",
  "source_sha256": "fb72aac548f61578bc9ba52448793b81bf61e37eff146f9519124d0794a4f4e5",
  "source_verified": true,
  "input_representation": "Complete deterministically normalised layout-aware structural Markdown",
  "representation_file": "D10_branch_C_normalised_markdown.md",
  "representation_sha256": "6fef221e9ac214f34ac330a7a63967290edeaa56e6f2f18e10b3cb0bd5ccc651",
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "structural_conversion_inherited_from_branch_B": true,
  "branch_B_regeneration_attempted": false,
  "normalisation_applied": true,
  "complete_source_document_retained": true,
  "source_scope_filtering_applied": false,
  "reference_values_used_for_transformation": false,
  "expected_record_cou

In [20]:
# ============================================================
# 19. Final artefact inventory and downloads
# ============================================================

artefacts = [
    PARENT_CHECK_PATH,
    NORMALISATION_CHECK_PATH,
    REPRESENTATION_PATH,
    REPRESENTATION_METADATA_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH,
    PRECHECK_PATH,
    RAW_RESPONSE_PATH,
    STRUCTURE_CHECK_PATH,
    EXPERIMENT_METADATA_PATH,
    EXPERIMENT_SUMMARY_PATH
]

if parsed_extraction_created:
    artefacts.append(
        PARSED_EXTRACTION_PATH
    )


print(
    "Final D10 Branch C artefacts:"
)

for path in artefacts:
    print(
        "-",
        path.name,
        "| exists:",
        path.exists()
    )


for path in artefacts:
    if path.exists():
        files.download(
            path
        )


Final D10 Branch C artefacts:
- D10_branch_C_parent_B_equivalence_check.json | exists: True
- D10_branch_C_normalisation_check.json | exists: True
- D10_branch_C_normalised_markdown.md | exists: True
- D10_branch_C_representation_metadata.json | exists: True
- D10_branch_C_prompt.txt | exists: True
- D10_branch_C_experiment_metadata_pre.json | exists: True
- D10_branch_C_pre_extraction_check.json | exists: True
- D10_branch_C_raw_response.txt | exists: True
- D10_branch_C_structure_check.json | exists: True
- D10_branch_C_experiment_metadata.json | exists: True
- D10_branch_C_experiment_summary.json | exists: True
- D10_branch_C_parsed_extraction.json | exists: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>